In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/sahilbhatia/pandax/dias_notebooks/nickwan/creating-player-stats-using-tracking-data/src/small_bench/checkpoints/post_cell_9.pickle

In [ ]:
%%cudf.pandas.profile
### cell 10 ###

# cell 10 – optimized for cuDF
# compute max LOS diff for offense and min LOS diff for defense, then average per player
off = (
    df[df['posTeam'] == 1]
      [['gameId','playId','nflId','los_diff']]
      .groupby(['gameId','playId','nflId'], as_index=False)
      .max()
)
def1 = (
    df[df['posTeam'] == 0]
      [['gameId','playId','nflId','los_diff']]
      .groupby(['gameId','playId','nflId'], as_index=False)
      .min()
)
# use GPU‐side concat instead of pandas append
los_diff = pd.concat([off, def1], ignore_index=True)
# average LOS diff per nflId
top = (
    los_diff[['nflId','los_diff']]
      .groupby('nflId', as_index=False)
      .mean()
)
# merge back onto snap_speed and rename columns
snap_speed = (
    snap_speed
      .merge(top)
      .rename(columns={
          's':'snap_s',
          'a':'snap_a',
          'los_diff':'snap_los_diff'
      })
)